recognize a person's face by comparing facial images to that of a
known person. The experimental dataset is uploaded at “Files”. There 40 subjects in this dataset and each
subject has ten images. The size of image is 112 x 92 pixels.

In [2]:
import numpy as np
import cv2
import os

In [3]:
def load_img(data_path):
    images = []
    labels = []
    for filename in sorted(os.listdir(data_path)):
        if filename.endswith('.png'):
            img_path = os.path.join(data_path, filename)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)  
            images.append(img.flatten())
            label = str(filename.split('_')[0])
            labels.append(label)

    return np.array(images), np.array(labels)

In [4]:
data_path = 'ATT'
images, labels = load_img(data_path)
print(f"Loaded {images.shape[0]} images with size {images.shape[1]}")

Loaded 400 images with size 10304


1NN: Do classification using KNN (1NN in this project) with 5-fold cross validation. Report
average accuracy.

In [6]:
def distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))


def custom_kfold_split(data, n_splits=5, random_state=1):
    np.random.seed(random_state)
    
    indices = np.arange(len(data))
    np.random.shuffle(indices)

    fold_sizes = len(data) // n_splits
    folds = [indices[i * fold_sizes: (i + 1) * fold_sizes] for i in range(n_splits)]
    
    for i in range(n_splits):
        test_indices = folds[i]
        train_indices = np.hstack([folds[j] for j in range(n_splits) if j != i])
        yield train_indices, test_indices


def knn_classification(images, labels, n_splits=5):
    # kf = KFold(n_splits=n_splits, shuffle=True, random_state=1)
    kf = custom_kfold_split(images, n_splits=n_splits)
    accuracies = []
    
    for train_index, test_index in kf:
        X_train, X_test = images[train_index], images[test_index]
        y_train, y_test = labels[train_index], labels[test_index]
        
        # 1NN
        y_pred = []
        for test_img in X_test:
            distances = [distance(test_img, train_img) for train_img in X_train]
            nearest_neighbor_idx = np.argmin(distances)
            y_pred.append(y_train[nearest_neighbor_idx])
            
            
        # Predict
        accuracy = np.sum(np.array(y_pred) == y_test) / len(y_test)
        accuracies.append(accuracy)
        
    # ave Accuracy
    avg_accuracy = np.mean(accuracies)
    return avg_accuracy

ave_accuracy = knn_classification(images, labels)
print(f"1NN Classification Average Accuracy: {ave_accuracy:.2f}") 

1NN Classification Average Accuracy: 0.99


1NN + PCA: In each cross validation, using PCA to reduce the dimensionality of images
(need to center the images when calculating PCA) to 100. Report average accuracy.

In [8]:
def pca(X_train, X_test, n_components):
    # Center
    mean_X_train = np.mean(X_train, axis=0)
    X_train_centered = X_train - mean_X_train
    X_test_centered = X_test - mean_X_train 
    
    covariance_matrix = np.cov(X_train_centered, rowvar=False)
    
    # Eigen decomposition
    eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)
    
    # Sort eigenvectors by eigenvalues in descending order
    sorted_indices = np.argsort(eigenvalues)[::-1]
    sorted_eigenvectors = eigenvectors[:, sorted_indices]
    
    # Select the top n_components eigenvectors
    principal_components = sorted_eigenvectors[:, :n_components]
    
    X_train_pca = np.dot(X_train_centered, principal_components)
    X_test_pca = np.dot(X_test_centered, principal_components)
    
    return X_train_pca, X_test_pca



def knn_pca_classification(images, labels, n_splits=5, n_components=100):
    kf = custom_kfold_split(images, n_splits=n_splits)
    accuracies = []
    
    for train_index, test_index in kf:
        X_train, X_test = images[train_index], images[test_index]
        y_train, y_test = labels[train_index], labels[test_index]
        
        # PCA
        X_train_pca, X_test_pca =  pca(X_train, X_test, n_components=n_components)
        
        # 1NN 
        y_pred = []
        for test_img in X_test_pca:
            distances = [distance(test_img, train_img) for train_img in X_train_pca]
            nearest_neighbor_idx = np.argmin(distances)
            y_pred.append(y_train[nearest_neighbor_idx])
            
            
        # Predict
        accuracy = np.sum(np.array(y_pred) == y_test) / len(y_test)
        accuracies.append(accuracy)
        
    avg_accuracy = np.mean(accuracies)
    return avg_accuracy

ave_accuracy = knn_pca_classification(images, labels)
print(f"1NN+PCA Classification Average Accuracy: {ave_accuracy:.2f}") 

1NN+PCA Classification Average Accuracy: 0.97


Resize images from 112 x 92 to 56 x 46 and repeat Task 2, compare the new results to the
results using un-resized images

In [11]:
def resize_img(images, new_size = (56,46)):
    resized_imges = []
    for img in images:
        img_2d = img.reshape(112,92)
        resized_img = cv2.resize(img_2d, new_size)
        resized_imges.append(resized_img.flatten())
    return np.array(resized_imges)

In [10]:
resized_images = resize_img(images)
ave_accuracy = knn_pca_classification(resized_images, labels)
print(f"1NN+resized image+PCA Classification Average Accuracy: {ave_accuracy:.2f}") 

1NN+resized image+PCA Classification Average Accuracy: 0.97


In [12]:
print("Resize images from 112 x 92 to 56 x 46 didn't make significent change accuracy")

Resize images from 112 x 92 to 56 x 46 didn't change a lot on accuracy
